# Modified Tarjan pathway reconstruction

This focused notebook tests pathway reconstruction using threshold stability, SCC-condensation DAG path enumeration, degree-preserving nulls, and downstream Schur/eigenmode analysis. Candidate paths are structural hypotheses, not proof of physiological propagation.

## Convention Lock

Every calculation follows Borst & Leibold:

$$\boxed{M_{ij}=\text{weight from presynaptic cell }j\text{ to postsynaptic cell }i}.$$

Rows are postsynaptic targets, columns are presynaptic sources, and feedforward ordering is lower triangular. If your CSV rows are presynaptic/source and columns are postsynaptic/target, set `MATRIX_ORIENTATION = "pre_rows_post_columns"`; the loader will transpose it once for analysis. If the CSV is already rows=post and columns=pre, set `MATRIX_ORIENTATION = "post_rows_pre_columns"`. Use `NORMALIZATION = "none"` to skip spectral-radius scaling.

## tl;dr

The canonical EC → DG → CA3c → CA1 paths are strong and survive through Q80, but modified Tarjan never places the top paths in that complete cell order. Degree-preserving nulls do not show significant enrichment of their scores or SCC separation. Stable paths nonetheless participate substantially in selected eigenmodes and Schur coordinates, and three-edge lesions rotate the slow Schur subspace more than they shift the leading eigenvalue.

## Setup and Data Checks

The setup cell below is the runtime control panel. Change `INPUT_MATRIX_PATH`, `INPUT_NETLIST_PATH`, `MATRIX_ORIENTATION`, and `NORMALIZATION` for each dataset. `INPUT_NETLIST_PATH` may be `None`; in that case orientation comes entirely from `MATRIX_ORIENTATION`.

The threshold grid is $q=0,5,\ldots,90$:

$$\theta_q=Q_q(\{|M_{ij}|:M_{ij}\ne0\}),\qquad M^{(q)}_{ij}=M_{ij}\mathbf 1(|M_{ij}|\ge\theta_q).$$

Null testing uses Q25, Q50, and Q75 with 100 directed degree-preserving rewires per level. Seeds are fixed.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
from modified_tarjan_pathway_analysis_script import run_analysis

# Runtime dataset controls
INPUT_MATRIX_PATH = "mij_matrix.csv"
INPUT_NETLIST_PATH = "mij_netlist.csv"  # Set to None if no netlist is available.

# Choose one:
# - "pre_rows_post_columns": input CSV rows are presynaptic/source, columns are postsynaptic/target.
# - "post_rows_pre_columns": input CSV rows are postsynaptic/target, columns are presynaptic/source.
# - "auto_from_netlist": use the optional netlist to choose the lower-residual orientation.
MATRIX_ORIENTATION = "pre_rows_post_columns"

# Choose "spectral_radius" or "none".
NORMALIZATION = "spectral_radius"
SPECTRAL_RADIUS_TARGET = 1.0
NETLIST_ORIENTATION_TOLERANCE = 1e-10

OUTPUT_DIR = Path("outputs/modified_tarjan_pathways")
N_NULL = 100

result = run_analysis(
    matrix_path=INPUT_MATRIX_PATH,
    netlist_path=INPUT_NETLIST_PATH,
    output_dir=OUTPUT_DIR,
    n_null=N_NULL,
    matrix_orientation=MATRIX_ORIENTATION,
    normalization=NORMALIZATION,
    spectral_radius_target=SPECTRAL_RADIUS_TARGET,
    netlist_orientation_tolerance=NETLIST_ORIENTATION_TOLERANCE,
)
display(pd.Series(result["run_summary"]["audit"], name="input audit").to_frame())

### Executed Result

The audit table above is computed from the runtime inputs in the setup cell. It records the matrix file, optional netlist file, requested input orientation, whether the matrix was transposed for analysis, and whether weights were normalized.

## 1. Modified-Tarjan threshold stability

At each threshold, Tarjan partitions the graph into maximal SCCs and contraction produces a DAG:

$$G^{(q)}\longrightarrow\mathcal C^{(q)}=G^{(q)}/\mathrm{SCC}.$$

Modified Tarjan orders cells inside recurrent blocks by reducing

$$\left(N_{\mathrm{upper}},\ \sum_{i<j}|M'_{ij}|(j-i)\right).$$

Path stability is

$$S(p)=\frac{\#\{q:p\text{ appears among the top paths at }q\}}{\#\{q\}}.$$

High stability means robustness to cutoff, not uniqueness or causality.

In [ ]:
summary = result["summary"]
display(summary)
display(Image(filename=str(OUTPUT_DIR/"figures/threshold_pathway_stability.png")))

### Observed result

The network remains one 85-cell SCC through Q5. The largest recurrent block declines to 74 cells at Q25, 58 at Q50, 17 at Q75, 5 at Q80, and singleton SCCs at Q90. Canonical three-edge paths survive through Q80 and disappear at Q85. Their best score remains 0.968 while all constituent edges survive because thresholding removes rather than rescales retained weights.

## 2. Condensation-DAG path enumeration

For inter-SCC edge $a\rightarrow b$,

$$W_{ab}=\sum_{j\in a}\sum_{i\in b}|M_{ij}|.$$

For DAG path $p=(c_0,\ldots,c_L)$,

$$s_{\mathrm{DAG}}(p)=\left(\prod_{\ell=0}^{L-1}W_{c_\ell c_{\ell+1}}\right)^{1/L}.$$

A dynamic-programming beam search retains the 100 highest-scoring paths of at most eight modules. EC → DG → CA3 → CA1 is annotated only afterward. Canonical cell paths use

$$s_{\mathrm{cell}}=(|w_{\mathrm{EC,DG}}w_{\mathrm{DG,CA3}}w_{\mathrm{CA3,CA1}}|)^{1/3}.$$

If pathway cells share an SCC, their internal sequence is unresolved by condensation topology.

In [ ]:
print("Stable unbiased condensation transitions/paths")
display(result["dag_stability"].head(20))
print("Stable canonical paths evaluated after discovery")
display(result["canonical_stability"].head(20))
display(Image(filename=str(OUTPUT_DIR/"figures/q50_condensation_dag.png")))

### Observed result

The most stable inter-SCC transition is **CA3 Pyramidal → CA1 Horizontal Axo Axonic**, appearing at 16 of 19 thresholds. Persistent transitions prominently involve CA3/CA3c pyramidal output to CA1 interneurons and DG Granule output to DG interneurons. The strongest canonical route is **MEC LV Pyramidal → DG Granule → CA3c Pyramidal → CA1 Perforant Path Associated QuadD**, with score 0.968 and stability 17/19 = 0.895.

None of the top canonical paths follows the full modified-Tarjan cell order at any retained threshold. The pathway is present as a weighted path but is not recovered as the network’s global cell-level ordering.

## 3. Degree-preserving pathway null tests

Each null uses directed double-edge swaps

$$a\rightarrow b,\ c\rightarrow d\mapsto a\rightarrow d,\ c\rightarrow b,$$

preserving in-degree, out-degree, edge count, and density. Signed weights are shuffled over rewired edges. For a larger-is-better statistic,

$$p=\frac{1+\sum_{b=1}^{B}\mathbf 1(T_b\ge T_{\mathrm{obs}})}{B+1},$$

with reversed inequality for smaller-is-better statistics. The tests cover SCC fragmentation, DAG-path score, canonical score, canonical SCC separation, and modified-Tarjan agreement. They test degree-sequence expectations, not physiological causality.

In [ ]:
null_tests = result["null_tests"]
display(null_tests)
display(Image(filename=str(OUTPUT_DIR/"figures/pathway_null_tests.png")))

### Observed result

No pathway-reconstruction statistic reached one-sided p < 0.05 at Q25, Q50, or Q75. At Q75, the observed largest SCC was 17 versus null mean 20.94 (p=0.248); best DAG-path score was 59.67 versus 50.36 (p=0.119); and the strongest canonical score was 0.968 versus 0.868 (p=0.307). Modified-Tarjan agreement for top canonical paths was zero and no better than null.

The pathway is therefore threshold-stable but not exceptional under this degree-preserving, weight-shuffled null. Stability and null enrichment answer different questions.

## 4. Downstream Schur and eigenmode relevance

For $A=M-I$, $Av_k=\lambda_kv_k$. Pathway participation is

$$\Pi_k^{\mathrm{eig}}(P)=\frac{\sum_{i\in P}|v_{ik}|^2}{\sum_i|v_{ik}|^2},\qquad
\Pi_k^{\mathrm{Schur}}(P)=\sum_{i\in P}|u_{ik}|^2.$$

After deleting the three pathway edges to form $A_{-P}$, we report

$$\Delta\alpha=\max\operatorname{Re}\lambda(A_{-P})-\max\operatorname{Re}\lambda(A),$$

plus the largest principal angle between baseline and lesioned six-dimensional slow Schur subspaces. Participation is association; edge deletion is a model perturbation, not a biological intervention.

In [ ]:
participation = result["participation"]
top_eigen = participation[(participation.path_rank==1)&(participation.basis=="eigen")].sort_values("participation",ascending=False)
top_schur = participation[(participation.path_rank==1)&(participation.basis=="schur")].sort_values("participation",ascending=False)
print("Strongest pathway: eigenmode participation")
display(top_eigen.head(8))
print("Strongest pathway: Schur-coordinate participation")
display(top_schur.head(8))
display(result["lesions"].head(20))
display(Image(filename=str(OUTPUT_DIR/"figures/pathway_modal_participation.png")))

### Observed result

The strongest canonical path contributes 20.1% of eigenmode 2, 14.3% of eigenmode 7, and 12.6% of each member of eigenmode pair 8/9. Its largest Schur-coordinate participation is 19.2% in coordinate 5, followed by 15.8% in coordinate 4 and 11.8% in coordinate 2. Removing its three edges shifts the spectral abscissa only to approximately $-4.93\times10^{-4}$ but rotates the slow Schur subspace by as much as 72.3°. Stable variants yield maximum rotations of roughly 19°–86°.

Thus these paths overlap meaningfully with several population patterns. Their edges alter slow-subspace orientation far more than the leading eigenvalue, although the marginal normalization and aggregate edge deletion require cautious interpretation.

## Takeaways

- Modified Tarjan identifies where direction is resolvable **between SCCs**; it cannot establish a total order inside recurrent blocks.
- The canonical trisynaptic path is strong and threshold-stable through Q80.
- Its full cell sequence is not reproduced by modified-Tarjan ordering.
- Degree-preserving nulls do not show exceptional path score or SCC separation.
- Stable condensation transitions emphasize CA3/CA3c-to-CA1 and DG-to-DG-interneuron edges.
- Stable canonical paths participate substantially in selected eigenmodes and Schur coordinates.
- Three-edge lesions rotate slow invariant subspaces much more than they shift the leading eigenvalue.

**Overall assessment: structurally reproducible but not null-enriched; dynamically associated but not uniquely established as the global information-flow ordering.**